# Lab de Limpeza - Camada Silver: Clientes

**Comentário:** Descreva aqui as regras de negócio identificadas durante a limpeza manual.

In [6]:
import pandas as pd
import s3fs

In [7]:
storage_options = {
    "key": "minio",
    "secret": "minio123",
    "client_kwargs": {"endpoint_url": "http://localhost:9000"}
}

In [8]:
fs = s3fs.S3FileSystem(**storage_options)
arquivos = fs.glob("bronze/clientes/**/*.csv")
arquivos

['bronze/clientes/ingestion_date=2026-04-29T00:30:04/clientes.csv']

In [9]:
df = pd.concat(
    [
        pd.read_csv(f"s3://{arquivo}", storage_options=storage_options) 
        for arquivo in arquivos
    ],
    ignore_index=True
)
df.head()

,id_cliente,nome_cliente,cnpj,canal,regiao,cidade,uf,representante,ativo,_ingested_at,_source_file
0,C0001,Distribuidora União,53.923.217/8673-19,Atacado,Triângulo Mineiro,Araguari,MG,Fernanda Souza,S,2026-04-29 00:30:04.128762,clientes.csv
1,C0002,Atacadão do Zé,60.114.370/9791-25,Food Service,Triângulo Mineiro,Uberlândia,MG,Juliana Costa,S,2026-04-29 00:30:04.128762,clientes.csv
2,C0003,Empório Santa Fé,57.210.791/4830-70,Atacado,Alto Paranaíba,Presidente Olegário,MG,Carlos Mendes,S,2026-04-29 00:30:04.128762,clientes.csv
3,C0004,Padaria da Praça,91.943.575/5951-93,Conveniência,Alto Paranaíba,Patos de Minas,MG,Juliana Costa,S,2026-04-29 00:30:04.128762,clientes.csv
4,C0005,Mercado do João,73.218.199/4846-78,Supermercado,Sul de Minas,Varginha,MG,Fernanda Souza,S,2026-04-29 00:30:04.128762,clientes.csv


In [10]:
cols_str = ["nome_cliente", "canal", "regiao", "cidade", "uf", "representante"]

for col in cols_str:
    df[col] = df[col].astype(str).str.strip()

In [11]:
df["_ingested_at"] = pd.to_datetime(df["_ingested_at"], errors="coerce")

In [12]:
df["uf"] = df["uf"].str.upper()

In [13]:
df["ativo"].unique()

<ArrowStringArray>
['S', '1', ' S', 'Sim', 'SIM', 's']
Length: 6, dtype: str

In [14]:
def normalize_ativo(value):
    if pd.isna(value):
        return None

    v = str(value).strip().lower()

    true_values = {"s", "sim", "1", "true", "t", "yes"}
    false_values = {"n", "nao", "não", "0", "false", "f", "no"}

    if v in true_values:
        return True
    elif v in false_values:
        return False
    else:
        return None 


df["ativo"] = df["ativo"].apply(normalize_ativo)

In [15]:
df = df[df["cnpj"].notna()]
df = df[df["nome_cliente"].notna()]

In [16]:
df

,id_cliente,nome_cliente,cnpj,canal,regiao,cidade,uf,representante,ativo,_ingested_at,_source_file
0,C0001,Distribuidora União,53.923.217/8673-19,Atacado,Triângulo Mineiro,Araguari,MG,Fernanda Souza,True,2026-04-29 00:30:04.128762,clientes.csv
1,C0002,Atacadão do Zé,60.114.370/9791-25,Food Service,Triângulo Mineiro,Uberlândia,MG,Juliana Costa,True,2026-04-29 00:30:04.128762,clientes.csv
2,C0003,Empório Santa Fé,57.210.791/4830-70,Atacado,Alto Paranaíba,Presidente Olegário,MG,Carlos Mendes,True,2026-04-29 00:30:04.128762,clientes.csv
3,C0004,Padaria da Praça,91.943.575/5951-93,Conveniência,Alto Paranaíba,Patos de Minas,MG,Juliana Costa,True,2026-04-29 00:30:04.128762,clientes.csv
4,C0005,Mercado do João,73.218.199/4846-78,Supermercado,Sul de Minas,Varginha,MG,Fernanda Souza,True,2026-04-29 00:30:04.128762,clientes.csv
...,...,...,...,...,...,...,...,...,...,...,...
115,C0200,Distribuidora União,53.923.217/8673-19,Atacado,Triângulo Mineiro,Araguari,MG,Fernanda Souza,True,2026-04-29 00:30:04.128762,clientes.csv
116,C0201,AtacadÃ£o do ZÃ©,60.114.370/9791-25,Food Service,Triângulo Mineiro,Uberlândia,MG,Juliana Costa,True,2026-04-29 00:30:04.128762,clientes.csv
117,C0202,Empório Santa Fé,57.210.791/4830-70,Atacado,Alto Paranaíba,Presidente Olegário,MG,Carlos Mendes,True,2026-04-29 00:30:04.128762,clientes.csv
118,C0203,Padaria da Praça,91.943.575/5951-93,Conveniência,Alto Paranaíba,Patos de Minas,MG,Juliana Costa,True,2026-04-29 00:30:04.128762,clientes.csv


In [17]:
df = df.sort_values("_ingested_at")

In [18]:
df = df.drop_duplicates(
    subset=["cnpj"],
    keep="last"
)

In [20]:
df.head()

,id_cliente,nome_cliente,cnpj,canal,regiao,cidade,uf,representante,ativo,_ingested_at,_source_file
87,C0088,Restaurante Três Irmãos,21.502.617/8491-40,Food Service,Noroeste de Minas,Unaí,MG,Fernanda Souza,True,2026-04-29 00:30:04.128762,clientes.csv
86,C0087,Bar Familiar,83.838.420/4110-30,Supermercado,Noroeste de Minas,João Pinheiro,MG,Juliana Costa,True,2026-04-29 00:30:04.128762,clientes.csv
85,C0086,Restaurante Três Irmãos,28.199.151/5754-59,Varejo Tradicional,Noroeste de Minas,Paracatu,MG,Rafael Lima,True,2026-04-29 00:30:04.128762,clientes.csv
84,C0085,Lanchonete Santa Fé,57.627.675/2739-50,Atacado,Alto Paranaíba,Patos de Minas,MG,Fernanda Souza,True,2026-04-29 00:30:04.128762,clientes.csv
83,C0084,Bar Três Irmãos,71.575.308/6573-87,Varejo Tradicional,Sul de Minas,Pouso Alegre,MG,Fernanda Souza,True,2026-04-29 00:30:04.128762,clientes.csv


In [22]:
try:
    df.to_parquet(
        path="s3://silver/clientes/",
        partition_cols=["regiao"],
        storage_options=storage_options,
        index=False,
        compression="snappy"
    )
    print("✓ Dados salvos com sucesso na camada Silver (MinIO)!")
except Exception as e:
    print(f"✗ Erro ao salvar na camada Silver: {e}")

✓ Dados salvos com sucesso na camada Silver (MinIO)!
